# 06. Diffusion Conditioning: ControlNet and IP-Adapter

- **Category:** 05_architectures / 05_generative_models
- **Lineage & Citations:** Lvmin Zhang et al., 2023 (*ControlNet*); Hu Ye et al., 2023 (*IP-Adapter*); Robin Rombach et al., 2022 (*Stable Diffusion*).
- **Purpose:** Formulates zero-convolution connections for locked-backbone spatial conditioning in ControlNet, alongside decoupled cross-attention branches for image-prompt adaptation in IP-Adapter.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Diffusion DDPM DDIM EDM and DiT](05_diffusion_ddpm_ddim_edm_and_dit.ipynb) $\to$ `🔗 Downstream Syntheses:` [Flow Matching Rectified Flow and Consistency Models](07_flow_matching_rectified_flow_and_consistency_models.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**
> - **Random Initialization of Zero-Convolutions:** Initializing $W_z \ne 0$ breaks the zero-noise identity invariant at step 0, corrupting pre-trained visual priors.
> - **Coupled Concatenation in IP-Adapter:** Concatenating image tokens with text tokens in the same cross-attention matrix causes image tokens to wash out text prompt semantics.

## Mathematical Foundations & Governing Equations

### 1. ControlNet Zero-Convolution Branch
$$\mathbf{y} = \mathcal{F}(\mathbf{x}; \Theta) + \mathcal{Z}\left(\mathcal{F}\left(\mathbf{x} + \mathcal{Z}(\mathbf{c}; \mathbf{W}_{z1}, \mathbf{b}_{z1}); \Theta_c\right); \mathbf{W}_{z2}, \mathbf{b}_{z2}\right)$$
With $\mathbf{W}_z = \mathbf{0}, \mathbf{b}_z = \mathbf{0}$, $\mathcal{Z}(\mathbf{x}) \equiv \mathbf{0}$ at step 0 while $\frac{\partial \mathcal{Z}}{\partial \mathbf{W}_z} = \mathbf{x}^\top \ne \mathbf{0}$.

### 2. IP-Adapter Decoupled Cross-Attention
$$\mathbf{Z}_{\text{out}} = \text{Attn}(\mathbf{Q}, \mathbf{K}_{\text{text}}, \mathbf{V}_{\text{text}}) + \lambda_{\text{img}} \text{Attn}(\mathbf{Q}, \mathbf{K}_{\text{img}}, \mathbf{V}_{\text{img}})$$

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/05_generative_models/06_diffusion_conditioning_controlnet_and_ip_adapter.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

x = weights["x"]
control_signal = weights["control_signal"]
backbone_w = weights["backbone_w"]
control_w = weights["control_w"]
z_in_w = weights["z_in_w"]
z_in_b = weights["z_in_b"]
z_out_w = weights["z_out_w"]
z_out_b = weights["z_out_b"]
scale = float(weights["scale"][0])
lambda_img = float(weights["lambda_img"][0])
print(f"Loaded {len(weights)} golden tensors on {env.device}, x: {list(x.shape)}, q: {list(weights['q'].shape)}")

In [ ]:
def controlnet_residual_block(x, control_signal, backbone_w, control_w, z_in_w, z_in_b, z_out_w, z_out_b):
    ctrl_in = x + (control_signal @ z_in_w + z_in_b)
    h_backbone = x @ backbone_w
    h_control = ctrl_in @ control_w
    ctrl_out = h_control @ z_out_w + z_out_b
    y = h_backbone + ctrl_out
    return y, ctrl_out

def ip_adapter_cross_attention(q, k_text, v_text, k_img, v_img, scale, lambda_img):
    scores_text = (q @ k_text.transpose(-2, -1)) * scale
    attn_text = F.softmax(scores_text, dim=-1) @ v_text
    scores_img = (q @ k_img.transpose(-2, -1)) * scale
    attn_img = F.softmax(scores_img, dim=-1) @ v_img
    return attn_text + lambda_img * attn_img

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
y_ctrl, ctrl_out = controlnet_residual_block(
    x, control_signal, backbone_w, control_w, z_in_w, z_in_b, z_out_w, z_out_b
)
err_ctrl = float(torch.max(torch.abs(y_ctrl - weights["y_ctrl"])))
assert err_ctrl < 1e-6, f"ControlNet output error: {err_ctrl:.2e}"
# Identity invariant assertion: at step 0 with zero-weights, ctrl_out must be identically 0
assert torch.all(ctrl_out == 0.0), "Zero-convolution at step 0 must output strictly zero!"

y_adapt, _ = controlnet_residual_block(
    x, control_signal, backbone_w, control_w, weights["z_in_w_adapt"], z_in_b, weights["z_out_w_adapt"], z_out_b
)
err_adapt = float(torch.max(torch.abs(y_adapt - weights["y_adapted"])))
assert err_adapt < 1e-6, f"ControlNet adapted output error: {err_adapt:.2e}"

ip_out = ip_adapter_cross_attention(
    weights["q"], weights["k_text"], weights["v_text"], weights["k_img"], weights["v_img"], scale, lambda_img
)
err_ip = float(torch.max(torch.abs(ip_out - weights["ip_out"])))
assert err_ip < 1e-6, f"IP-Adapter output error: {err_ip:.2e}"
print(f"Parity Verified: ControlNet Step0={err_ctrl:.2e}, Adapted={err_adapt:.2e}, IP-Adapter={err_ip:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_ctrl = benchmark_fn(lambda: controlnet_residual_block(
    x, control_signal, backbone_w, control_w, weights["z_in_w_adapt"], z_in_b, weights["z_out_w_adapt"], z_out_b
), iters=50, name="ControlNet Block")
b_ip = benchmark_fn(lambda: ip_adapter_cross_attention(
    weights["q"], weights["k_text"], weights["v_text"], weights["k_img"], weights["v_img"], scale, lambda_img
), iters=50, name="IP-Adapter Decoupled Attn")
print(format_benchmark_table([b_ctrl, b_ip]))

# Visualization: Backbone Feature vs Injected ControlNet Feature Map
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].imshow((x @ backbone_w)[0].detach().numpy(), cmap="magma", aspect="auto")
ax[0].set_title("Frozen Backbone Activations")
ax[1].imshow(y_adapt[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[1].set_title("ControlNet Conditioned Output")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse vs Corrected Implementation: Random Noise vs Zero-Init in ControlNet
# Random initialization injects massive uncalibrated noise into pre-trained features!
w_random = torch.randn_like(z_out_w) * 0.5
y_noisy, _ = controlnet_residual_block(
    x, control_signal, backbone_w, control_w, w_random, z_in_b, w_random, z_out_b
)
h_backbone = x @ backbone_w
dist_zero = float(torch.norm(y_ctrl - h_backbone))
dist_noisy = float(torch.norm(y_noisy - h_backbone))
print(f"L2 Drift from Backbone with Zero-Init: {dist_zero:.6f}")
print(f"L2 Drift from Backbone with Random-Init: {dist_noisy:.4f}")
assert dist_zero == 0.0, "Zero-init must cause exactly 0 drift from pre-trained backbone!"
assert dist_noisy > 1.0, "Random-init corrupts backbone features!"
print("Demonstrated requirement of zero-convolution initialization in ControlNet.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Zero-Init Preserves Generative Priors:** Initializing adapter projections to zero ensures identity transformation at training step 0.
2. **Decoupled Attention Eliminates Semantic Washout:** IP-Adapter separates text and image cross-attentions to prevent modality interference.
3. **Non-Destructive Transfer Learning:** Freezing the foundation model locks core generalisation while side-branches learn domain tasks.

---
### Cross-Domain Generalisation Matrix
- **Structural Conditioning in Video VFX:** ControlNet guides diffusion with camera motion matrices and depth maps.
- **Identity-Preserving Avatars:** IP-Adapter transfers face IDs and style references across variable text prompts.
- **Robotics Visual Policy Grounding:** Condition policy rollouts on segmented scene affordance masks.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_diffusion_conditioning_controlnet_and_ip_adapter.md](../../../markdown/05_architectures/05_generative_models/06_diffusion_conditioning_controlnet_and_ip_adapter.md)
- **PyTorch 2.x Module:** [06_diffusion_conditioning_controlnet_and_ip_adapter.py](../../../pytorch2/05_architectures/05_generative_models/06_diffusion_conditioning_controlnet_and_ip_adapter.py)
- **Functional JAX Module:** [06_diffusion_conditioning_controlnet_and_ip_adapter.py](../../../jax/05_architectures/05_generative_models/06_diffusion_conditioning_controlnet_and_ip_adapter.py)
- **Candle Rust Module:** [06_diffusion_conditioning_controlnet_and_ip_adapter.rs](../../../candle/05_architectures/05_generative_models/06_diffusion_conditioning_controlnet_and_ip_adapter.rs)
- **Hardware Kernel:** [06_diffusion_conditioning_controlnet_and_ip_adapter.rs](../../../kernels/05_architectures/05_generative_models/06_diffusion_conditioning_controlnet_and_ip_adapter.rs)
- **Universal Golden SafeTensors:** [06_diffusion_conditioning_controlnet_and_ip_adapter.safetensors](../../../weights/05_architectures/05_generative_models/06_diffusion_conditioning_controlnet_and_ip_adapter.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 05. Diffusion DDPM DDIM EDM and DiT](05_diffusion_ddpm_ddim_edm_and_dit.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 07. Flow Matching Rectified Flow and Consistency Models](07_flow_matching_rectified_flow_and_consistency_models.ipynb) &rarr;